# 03 — Sobol versus pseudo-random

Monte Carlo error with $N$ i.i.d. points decays like $N^{-1/2}$, whatever the dimension. A
*low-discrepancy* sequence fills the unit cube more evenly than chance, and the Koksma–Hlawka
inequality bounds the integration error by (discrepancy) × (variation of the integrand), where the
discrepancy of the best sequences is $O\big((\log N)^d / N\big)$ — asymptotically $N^{-1}$.

## 1. Van der Corput and Sobol

In one dimension, reverse the binary digits of $n$ around the point: $n = 1, 2, 3, 4 \mapsto 0.5, 0.25, 0.75, 0.125$.
Every block of $2^m$ consecutive points hits every dyadic interval of length $2^{-m}$ exactly once.
Sobol extends this to $d$ dimensions with one set of *direction numbers* $v_1, v_2, \dots$ per dimension:

$$
x_n = \bigoplus_{k:\ \text{bit } k \text{ of } n} v_k \qquad (\text{XOR of the direction numbers selected by the bits of } n).
$$

In Gray-code order ($g(n) = n \oplus (n \gg 1)$) consecutive $g$'s differ in one bit, so
$x_{n} = x_{n-1} \oplus v_{c}$ with $c$ the position of the lowest zero bit of $n-1$: one XOR per dimension per point.

The direction numbers come from a primitive polynomial per dimension (degree $s$, coefficients $a$)
and $s$ initial odd integers $m_1..m_s$ (Joe & Kuo 2008 chose them to optimise two-dimensional
projections):
$$
v_k = v_{k-s} \oplus (v_{k-s} \gg s) \oplus \bigoplus_{i=1}^{s-1} a_i\, v_{k-i}, \qquad k > s .
$$

### Worked example by hand: dimension 3, $s = 2$, $a = 1$, $m = (1, 3)$

$v_1 = 1 \cdot 2^{-1} = 0.5$, $v_2 = 3 \cdot 2^{-2} = 0.75$,
$v_3 = v_1 \oplus (v_1 \gg 2) \oplus v_2 = 0.100_2 \oplus 0.001_2 \oplus 0.110_2 = 0.011_2 = 0.375$.
Gray-code walk: $x_1 = v_1 = 0.5$; $n=2$: lowest zero bit of 1 is bit 1 → $x_2 = 0.5 \oplus 0.75 = 0.25$;
$n=3$: bit 0 → $0.75$; $n=4$: lowest zero bit of $3 = 11_2$ is bit 2 → $0.75 \oplus 0.375 = 0.625$.
So dimension 3 runs $0, 0.5, 0.25, 0.75, 0.625, 0.125, 0.875, 0.375$ — exactly the reference output of
Joe & Kuo's `sobol.cc`, and the golden in `test_sobol.cpp`.

## 2. Two conventions that must agree with the rest of the library

* Point 0 is the origin. Anything feeding an inverse normal cdf must skip it (`Sobol(dim, skip=1)`).
* Low dimensions are the well-distributed ones; the Brownian bridge therefore assigns
  dimension 1 to the terminal value of the path, dimension 2 to the midpoint, and so on.

In [ ]:
import _util
from _util import run_demo
import matplotlib.pyplot as plt
import numpy as np

pts = run_demo("rl_demo_sobol", "points", 512, 2)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, gen in zip(axes, ["sobol", "mt"]):
    p = pts[pts["generator"] == gen]
    ax.scatter(p["x1"], p["x2"], s=6)
    ax.set_title(f"{gen}: 512 points in 2-D"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### The integration test: $\int_{[0,1]^4} e^{-|x|_1}\,dx = (1 - e^{-1})^4$

Smooth, bounded variation, four dimensions: the textbook case where Sobol should show a slope
steeper than $-1/2$ on a log–log plot. The MT error is averaged over 16 seeds so the comparison is
against the *expected* MC error, not one lucky draw.

In [ ]:
err = run_demo("rl_demo_sobol", "integrate", 18, 4)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(err["N"], err["mtErr"], "o-", label="MT19937-64 (mean |error| over 16 seeds)")
ax.loglog(err["N"], err["sobolErr"], "s-", label="Sobol (Joe–Kuo), origin skipped")
ref = err["mtErr"].iloc[0] * (err["N"].iloc[0] / err["N"]) ** 0.5
ax.loglog(err["N"], ref, "k--", lw=0.8, label=r"$N^{-1/2}$ reference")
ref1 = err["sobolErr"].iloc[0] * (err["N"].iloc[0] / err["N"]) ** 1.0
ax.loglog(err["N"], ref1, "k:", lw=0.8, label=r"$N^{-1}$ reference")
ax.set_xlabel("N points"); ax.set_ylabel("|error|"); ax.legend(); ax.set_title(r"$\int_{[0,1]^4} e^{-|x|_1}$: Sobol vs pseudo-random")
plt.tight_layout()
from pathlib import Path
out = Path(_util._ROOT) / "docs" / "img" / "sobol_vs_mt.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=130)
print("saved", out)
plt.show()

In [ ]:
# Fitted slopes over the last decade of N.
tail = err[err["N"] >= 2**12]
for col in ["mtErr", "sobolErr"]:
    slope = np.polyfit(np.log(tail["N"]), np.log(tail[col]), 1)[0]
    print(f"{col}: error ~ N^{slope:.2f}")